# Structural break tests

Chow tests reported in Table 3.1.

In [1]:
from pathlib import Path
import os

# Locate the bundle from either its root or the notebooks directory.
_here = Path.cwd().resolve()
_ROOT = next((p for p in [_here, *_here.parents]
              if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if _ROOT is None:
    raise FileNotFoundError("Open this notebook inside the Tesi_codice_consegna folder.")
_DATA_DIR = _ROOT / "data"
_OUTPUT_DIR = _ROOT / "results" / "rerun" / "02_structural_breaks"
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)
print("Data:", _DATA_DIR)
print("New outputs:", _OUTPUT_DIR)


Data: /Users/macbook/Desktop/Tesi Master/Tesi_codice_consegna/data
New outputs: /Users/macbook/Desktop/Tesi Master/Tesi_codice_consegna/results/rerun/02_structural_breaks


## Chow tests

Original break dates, OLS specification, and testing calculation retained; paths only changed.

Source: `AAVE v2/chow test.ipynb`, original cell 0 (zero-based).

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

# Assumo tu abbia già:
df = pd.read_csv(str(_DATA_DIR / 'aavebtc.csv'), parse_dates=["date"], dayfirst=True)
# 1. Definisci breakpoints
break1 = '2022-02-01'  # Fine ZIRP
break2 = '2023-08-01'  # Inizio Plateau

# 2. Crea dummy per regimi
df['regime'] = 0
df.loc[df['date'] >= break1, 'regime'] = 1
df.loc[df['date'] >= break2, 'regime'] = 2

# 3. CHOW TEST - metodo manuale (più chiaro)
def chow_test(df, y_col, x_col, break_date):
    """
    H0: no structural break at break_date
    H1: coefficients change after break_date
    """
    # Split data
    df1 = df[df['date'] < break_date].copy()
    df2 = df[df['date'] >= break_date].copy()
    
    # Prepare data
    y_full = df[y_col]
    X_full = sm.add_constant(df[x_col])
    
    y1 = df1[y_col]
    X1 = sm.add_constant(df1[x_col])
    
    y2 = df2[y_col]
    X2 = sm.add_constant(df2[x_col])
    
    # Estimate 3 models
    model_full = sm.OLS(y_full, X_full).fit()
    model1 = sm.OLS(y1, X1).fit()
    model2 = sm.OLS(y2, X2).fit()
    
    # Get RSS
    rss_full = model_full.ssr
    rss_1 = model1.ssr
    rss_2 = model2.ssr
    rss_split = rss_1 + rss_2
    
    # Compute F-statistic
    k = X_full.shape[1]  # number of parameters
    n = len(df)
    n1 = len(df1)
    n2 = len(df2)
    
    F = ((rss_full - rss_split) / k) / (rss_split / (n - 2*k))
    
    # P-value
    p_value = 1 - stats.f.cdf(F, k, n - 2*k)
    
    return {
        'F_statistic': F,
        'p_value': p_value,
        'df1': k,
        'df2': n - 2*k,
        'RSS_full': rss_full,
        'RSS_split': rss_split,
        'n_obs': n,
        'significant': p_value < 0.05
    }

pol = "3pool"
ir = "3m ir"
# 4. RUN TESTS
result_break1 = chow_test(df, pol, ir, break1)
result_break2 = chow_test(df[df['date'] >= break1], pol, ir, break2)

# 5. PRINT RESULTS
print("=" * 60)
print("CHOW TEST - Break at February 2022 (End of ZIRP)")
print("=" * 60)
print(f"F-statistic: {result_break1['F_statistic']:.2f}")
print(f"p-value: {result_break1['p_value']:.4f}")
print(f"Significant at 5%: {'YES ✓' if result_break1['significant'] else 'NO ✗'}")

print("\n" + "=" * 60)
print("CHOW TEST - Break at August 2023 (Start of Plateau)")
print("=" * 60)
print(f"F-statistic: {result_break2['F_statistic']:.2f}")
print(f"p-value: {result_break2['p_value']:.4f}")
print(f"Significant at 5%: {'YES ✓' if result_break2['significant'] else 'NO ✗'}")


02_structural_breaks.ipynb:cell3:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
CHOW TEST - Break at February 2022 (End of ZIRP)
F-statistic: 2.81
p-value: 0.0606
Significant at 5%: NO ✗

CHOW TEST - Break at August 2023 (Start of Plateau)
F-statistic: 6.96
p-value: 0.0010
Significant at 5%: YES ✓
